### Traditional Spectrum Estimation Using the Periodogram

The periodogram is a simple method to estimate the power density spectrum
$S_{XX}(\mathrm{e}^{j\Omega})$ based on $N$ samples of a stationary process $X(k)$.

$$
\mathrm{Per}_N^X(\Omega) := \frac{1}{N} \left| X(e^{j\Omega}) \right|^2,
\quad \text{with} \quad X(e^{j\Omega}) = \mathcal{F}{ X(k) }, \quad k=0,\dots,N-1
$$

1. Create two model processes $X_1(k), X_2(k)$ with length $N=2^{12}$ by linear filtering of a white Gaussian and zero mean noise process $N(k)$ with power one using filters $H_1(z)$ and $H_2(z)$:

$$
H_1(z) = 1 - 4 z^{-1} + 6 z^{-2} - 4 z^{-3} + z^{-4}, \quad
H_2(z) = \frac{1}{1 + 0.8, z^{-1}}.
$$

2. Plot the "true" power density spectrum $S_{X_i X_i}(\mathrm{e}^{j\Omega})$ for both filters.

3. Calculate the periodograms $\mathrm{Per}_N^{X_i}(\Omega)$ for $N = 2^6, 2^8, 2^{10}$ and $N=2^{12}$ samples for both processes.
   Why does the variance of the estimation not decrease if the number of samples $N$ increases?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from helper_functions import lper
# BEGIN SOLUTION
N = 2**12
N_vor = 500
n = np.random.randn(N + N_vor)

# Coefficient vector for filters H1 and H2, filtering with N
h1 = [1, -4, 6, -4, 1]
x1_all = np.convolve(h1, n)
x1 = x1_all[N_vor:N_vor + N]

h2 = [1, 0.8]

# Filter H2 (IIR)
x2_all = lfilter([1], h2, n)
x2 = x2_all[N_vor:N_vor + N]

# True power spectral density for the two filters
NFFT = 2**12

Sx1x1 = np.abs(np.fft.fft(h1, NFFT))**2
axis_Sx1 = [0, 1, 0, 2 * max(Sx1x1)]

Sx2x2 = 1 / (np.abs(np.fft.fft(h2, NFFT))**2)
axis_Sx2 = [0, 1, 0, 1.5 * max(Sx2x2)]

fT = np.arange(0, 1, 1/NFFT)

# Graphical representation
plt.figure()
plt.plot(fT, Sx1x1)
plt.axis(axis_Sx1)
plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
plt.ylabel(r'$S_{XX}(e^{j\cdot\Omega})$')
plt.title(r'True Power Spectral Density $S_{X1X1}$ of $x_1$')

plt.figure()
plt.plot(fT, Sx2x2)
plt.axis(axis_Sx2)
plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
plt.ylabel(r'$S_{XX}(e^{j\cdot\Omega})$')
plt.title(r'True Power Spectral Density $S_{X2X2}$ of $x_2$')

input('....Press RETURN for the next task:')

# Draw periodograms and true power spectral densities
for k in [2**6, 2**8, 2**10, 2**12]:
    Sxx = lper(x1[:k], NFFT)
    plt.figure()
    plt.plot(fT, Sxx)
    plt.plot(fT, Sx1x1, '--')
    plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
    plt.ylabel('Per(Om)')
    plt.title(f'Periodogram of $S_{{X1X1}}$, N={k}')
    plt.axis(axis_Sx1)
    plt.show()

input('....Press RETURN to continue:')

for k in [2**6, 2**8, 2**10, 2**12]:
    Sxx = lper(x2[:k], NFFT)
    plt.figure()
    plt.plot(fT, Sxx)
    plt.plot(fT, Sx2x2, '--')
    plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
    plt.ylabel('Per(Om)')
    plt.title(f'Periodogram of $S_{{X2X2}}$, N={k}')
    plt.axis(axis_Sx2)
    plt.show()
# END SOLUTION

### Solution: Traditional Spectrum Estimation Using the Periodogram

a) Given an input signal vector `n = [n_0, ..., n_{N-1}]`, the Python function

```python
from scipy.signal import lfilter

xi = lfilter(bi, ai, n)
```

calculates the output signal

```
xi = [x_{0,i}, ..., x_{N-1,i}]
```

of the ARMA model

$$
H_i(z) = \frac{B_i(z)}{A_i(z)} = \frac{\sum_{\mu=0}^{m} b_{\mu,i} z^{-\mu}}{\sum_{\nu=0}^{n} a_{\nu,i} z^{-\nu}},
$$

where the MA-coefficients $b_{\mu,i}$ and the AR-coefficients are given by vectors `bi` and `ai`.

For $H_1(z)$: `b1 = [1, -4, 6, -4, 1]` and `a1 = 1`
For $H_2(z)$: `b2 = 1` and `a2 = [1, 0.8]`

---

b) For stationary white noise with unit variance $\sigma_N^2 = 1$, the true power density spectra of the output signal $x_i(k)$ are:

$$
S_{X_i X_i}(e^{j\Omega}) = \sigma_N^2 \cdot |H_i(e^{j\Omega})|^2 = \left| \frac{B_i(e^{j\Omega})}{A_i(e^{j\Omega})} \right|^2 = \frac{| \mathcal{F}{b_i(k)} |^2}{| \mathcal{F}{a_i(k)} |^2}.
$$

To portray the true power density spectra $S_{X_1 X_1}(e^{j\Omega})$ and $S_{X_2 X_2}(e^{j\Omega})$, the discrete-time Fourier transform $\mathcal{F}{\cdot}$ is approximated by a $2^{12}$-point FFT.

<img src="../pngs/ldsv8a02_sx1x1.png" width="250"> <img src="../pngs/ldsv8a02_sx2x2.png" width="250">

---

c) The following figures show the periodograms $\mathrm{Per}_N^{X_i}(\Omega)$ with $N=2^6=64$, $2^8=256$, $2^{10}=1028$ and $N=2^{12}=4096$ data values of each sample process $X_1(k)$ and $X_2(k)$.

<img src="../pngs/ldsv8a02_perx1N64.png" width="250"> <img src="../pngs/ldsv8a02_perx1N256.png" width="250"> <img src="../pngs/ldsv8a02_perx1N1024.png" width="250"> <img src="../pngs/ldsv8a02_perx1N4096.png" width="250">

<img src="../pngs/ldsv8a02_perx2N64.png" width="250"> <img src="../pngs/ldsv8a02_perx2N256.png" width="250"> <img src="../pngs/ldsv8a02_perx2N1024.png" width="250"> <img src="../pngs/ldsv8a02_perx2N4096.png" width="250">

The discrete-time Fourier transform (DTFT) $\mathcal{F}{\cdot}$ is again approximated through a $2^{12}$-point FFT. The true spectral power density $S_{X_i X_i}(e^{j\Omega})$ is shown by a dashed line.
It is recognizable that the spectral estimation using the periodogram gives unsatisfying results due to the variance

$$
\mathrm{Var}{ \mathrm{Per}*N^{X_i}(\Omega) } \sim S*{X_i X_i}(e^{j\Omega})^2.
$$

Only in the frequency intervals where the true power density $S_{X_i X_i}(e^{j\Omega})$ vanishes, the periodogram delivers zero (with disappearing variance). In frequency regions with spectral components, the periodogram permits no prediction about $S_{X_i X_i}(e^{j\Omega})$, because the estimation is devalued by excessively high variance. Increasing $N$ also does not change this behavior.
The bias behavior improves if $N$ increases.